# Language Translation

This tutorial is based on the language translation example of the PyTorch [official examples set](https://github.com/pytorch/examples/tree/main/language_translation). This example shows how one might use transformers for language translation. In particular, this implementation is loosely based on the [Attention is All You Need paper](https://arxiv.org/abs/1706.03762).

Once the code is ready, we can execute it in the platform. As all the operations of the platform, we start from defining the context of our experiments and executions, or **project**. Project is a logical container for data, artifacts, models, executable operations, and their executions.

### Define the context

It is possible to create the project via platform UI or programmatically using the platform SDK:


In [ ]:
import digitalhub as dh

project = dh.get_or_create_project("translation-tutorial")

### Prepare and register the input data

The next step is to prepare and explicitly register the input data necessary for our experiment. Making it explicit allows for better tracability and reproducibility of the experiments. We will use the same data as in the original code. Download it locally for further processing.

- training: [https://raw.githubusercontent.com/neychev/small_DL_repo/master/datasets/Multi30k/training.tar.gz](https://raw.githubusercontent.com/neychev/small_DL_repo/master/datasets/Multi30k/training.tar.gz)
- validation: [https://raw.githubusercontent.com/neychev/small_DL_repo/master/datasets/Multi30k/validation.tar.gz](https://raw.githubusercontent.com/neychev/small_DL_repo/master/datasets/Multi30k/validation.tar.gz)

It is possible to upload the data artifacts via UI or programmatically:


In [ ]:
project.log_artifact("train-data", kind="artifact", source="training.tar.gz")

In [ ]:
project.log_artifact("validation-data", kind="artifact", source="validation.tar.gz")

### Register function

Next, we need to describe and register our training function. Our code is pure Python Job application, so we can use `python` runtime for this purpose. Again, it is possible to do it via UI or programmatically:


In [ ]:
func = project.new_function(
    "train-translator",
    kind="python",
    python_version="PYTHON3_12",
    code_src="git+https://github.com/scc-digitalhub/digitalhub-tutorials",
    handler="torch-translation-tutorial.wrapper:train",
    requirements=["torch==2.3.0", "torchtext==0.18.0", "torchdata==0.9.0", "spacy==3.8.14", "portalocker==3.2.0", "click>=8.2.1", "digitalhub==0.15.5"]
)

Note the reference to the git repository where the training function is located: by default it will point out to the **current** version of the code in the main branch. So each time the function is executed, the latest version of the code will be used. However, it is possible to point to a specific commit, branch, or tag using the standard git reference syntax (e.g., `git+https://github.com/scc-digitalhub/digitalhub-tutorials#main`).

If the code is at the private repository, it is possile to [configure the credentials](https://scc-digitalhub.github.io/docs/tasks/code-source/#remote-git-repository) to access it, using environment variables or [secrets](https://scc-digitalhub.github.io/docs/tasks/secrets/), such as, e.g, `GITHUB_TOKEN`. 

The entry point is defined as `handler` attribute composed of python path to the containing module and the function name.

The dependencies are listed explicitly with their versions.


### Executing the function locally

The function can be executed via UI or programmatically. First, we run the function locally (directly in the current environment). Note that for this the required dependencies should have been already installed.

Note that 
- hyper parameters are passed as `parameters` dictionary;
- input data entities are passed as `inputs` dictionary using the artifact keys as values (sort of unique artifact identifiers);


In [ ]:
run = func.run(
    action="job",
    inputs={
        "training_data": project.get_artifact("train-data").key,
        "validation_data": project.get_artifact("validation-data").key
    },
    parameters={
        "epochs": 1,
        "backend": "gpu"
    },
    local_execution=True
)

### Building a container image

The build operation may be triggered by the UI or programmatically. The build is necessary to run the function remotely (on the platform cluster as a Kubernetes Job).

In [ ]:
func.run(action="build")

This will result in the container image being created within the platform cluster, the image will be published in the internal image registry, and the function will be associated with the image. If the list of the dependencies changes, the container image should be rebuild. The images with the same dependencies and base image may be reused across different functions and executions.

If the build operation is not performed, the execution will fail as the base container image does not include the required dependencies.

Please note that it is possible to add further functionality to the container image passing additional build instructions (e.g., to download and pre-install the language models).


### Executing the function in the cluster

In this example the function is executed remotely (`local_execution=False`). Note that the function is executed on the cluster with the specified hardware profile (e.g., "1xV100" - 1 node with 1 V100 GPU). The list of profiles depend on the cluster configuration of the platform. Indeed, when the function is executed locally, the profile parameter is ignored.

In [ ]:
run = func.run(
    action="job",
    inputs={
        "training_data": project.get_artifact("train-data").key,
        "validation_data": project.get_artifact("validation-data").key
    },
    parameters={
        "epochs": 10,
        "backend": "gpu"
    },
    envs=[{"name": "DHCORE_LOG_LEVEL", "value": "DEBUG"}],
    profile="1xV100",
    local_execution=False
)

## Using the model for inference

Once model is trained, we can test it for inference. We start from downloading the model weights locally and create the inference from them.

In [ ]:
model = project.get_model("translator-model")
model.spec.parameters

In [ ]:
model.download("./model/", overwrite=True)

In [ ]:
from src.model import Translator # Our model
from src.data import get_data, create_mask, generate_square_subsequent_mask # Loading data and data preprocessing

import torch 

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

opts = type("Namespace", (), {})()
setattr(opts, "src", model.spec.parameters['src'])
setattr(opts, "tgt", model.spec.parameters['tgt'])
setattr(opts, "batch", model.spec.parameters['batch'])
setattr(opts, "train_file", "training.tar.gz")
setattr(opts, "valid_file", "validation.tar.gz")

_, _, src_vocab, tgt_vocab, src_transform, _, special_symbols = get_data(opts)

src_vocab_size = len(src_vocab)
tgt_vocab_size = len(tgt_vocab)

# Create model
translator = Translator(
    num_encoder_layers=model.spec.parameters['enc_layers'],
    num_decoder_layers=model.spec.parameters['dec_layers'],
    embed_size=model.spec.parameters['embed_size'],
    num_heads=model.spec.parameters['attn_heads'],
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    dim_feedforward=model.spec.parameters['dim_feedforward'],
    dropout=model.spec.parameters['dropout']
).to(DEVICE)

# Load in weights
translator.load_state_dict(torch.load("./model/best.pt"))

# Set to inference
translator.eval()

In [ ]:
sentence = "Ich verstehe nicht."

def greedy_decode(model, src, src_mask, max_len, start_symbol, end_symbol):

    # Move to device
    src = src.to(DEVICE)
    src_mask = src_mask.to(DEVICE)

    # Encode input
    memory = model.encode(src, src_mask)

    # Output will be stored here
    ys = torch.ones(1, 1).fill_(start_symbol).type(torch.long).to(DEVICE)

    # For each element in our translation (which could range up to the maximum translation length)
    for _ in range(max_len-1):

        # Decode the encoded representation of the input
        memory = memory.to(DEVICE)
        tgt_mask = (generate_square_subsequent_mask(ys.size(0), DEVICE).type(torch.bool)).to(DEVICE)
        out = model.decode(ys, memory, tgt_mask)

        # Reshape
        out = out.transpose(0, 1)

        # Covert to probabilities and take the max of these probabilities
        prob = model.ff(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.item()

        # Now we have an output which is the vector representation of the translation
        ys = torch.cat([ys, torch.ones(1, 1).type_as(src.data).fill_(next_word)], dim=0)
        if next_word == end_symbol:
            break

    return ys


# Convert to tokens
src = src_transform(sentence).view(-1, 1)
num_tokens = src.shape[0]

src_mask = (torch.zeros(num_tokens, num_tokens)).type(torch.bool)

# Decode
tgt_tokens = greedy_decode(
    translator, src, src_mask, max_len=num_tokens+5, start_symbol=special_symbols["<bos>"], end_symbol=special_symbols["<eos>"]
).flatten()

# Convert to list of tokens
output_as_list = list(tgt_tokens.cpu().numpy())

# Convert tokens to words
output_list_words = tgt_vocab.lookup_tokens(output_as_list)

# Remove special tokens and convert to string
translation = " ".join(output_list_words).replace("<bos>", "").replace("<eos>", "")

print(translation)
